# ViT Unfreeze Comparison (2 / 4 / 6)

This notebook keeps the dataset split, augmentation, and optimization setup fixed, and compares stage-2 fine-tuning with the last 2, 4, or 6 ViT transformer blocks unfrozen.


In [ ]:
# Local/WSL workflow: no Colab drive mount needed.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    mixed_precision.set_global_policy("mixed_float16")

print("TF version:", tf.__version__)
print("GPU:", gpus)
print("Mixed precision policy:", mixed_precision.global_policy())

import matplotlib.pyplot as plt
import numpy as np
import os
import random
import keras_hub
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


In [ ]:
def count_files_by_class(root_dir):
    counts = {}
    for class_name in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        counts[class_name] = sum(
            1
            for file_name in os.listdir(class_dir)
            if file_name.lower().endswith((".jpg"))
        )
    return counts


def summarize_labels(ds, class_names):
    counts = Counter()
    for _, labels in ds:
        labels = labels.numpy().astype("int32").reshape(-1)
        counts.update(labels.tolist())
    return {class_names[idx]: counts.get(idx, 0) for idx in range(len(class_names))}


def collect_predictions(ds, model):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0).reshape(-1)
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(pb)
    y_true = np.concatenate(y_true).astype("int32")
    y_prob = np.concatenate(y_prob)
    return y_true, y_prob


In [ ]:
data_dir = "/mnt/e/DDSM/ROI"
assert os.path.exists(data_dir), f"Dataset path not found: {data_dir}"
print("Using data_dir:", data_dir)
raw_class_counts = count_files_by_class(data_dir)
print("Raw file counts by class:", raw_class_counts)


In [ ]:
img_size = (224, 224)
batch_size = 16
seed = 123
AUTOTUNE = tf.data.AUTOTUNE

os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
keras.utils.set_random_seed(seed)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
)

class_names = train_ds.class_names
print("Classes:", class_names)
print(f"Positive class for AUC/ROC: {class_names[1]}")

temp_card = tf.data.experimental.cardinality(temp_ds).numpy()
if temp_card < 2:
    raise ValueError(f"Validation split is too small: only {temp_card} batch(es). Reduce batch_size or use more data.")

val_batches = max(1, temp_card // 2)
test_batches = temp_card - val_batches
if test_batches == 0:
    raise ValueError("Test split is empty. Reduce batch_size or use more data.")

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.prefetch(AUTOTUNE)

train_ds_prep = prepare(train_ds, shuffle=True)
val_ds_prep = prepare(val_ds)
test_ds_prep = prepare(test_ds)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())

train_label_counts = summarize_labels(train_ds, class_names)
val_label_counts = summarize_labels(val_ds, class_names)
test_label_counts = summarize_labels(test_ds, class_names)

print("Train label counts:", train_label_counts)
print("Validation label counts:", val_label_counts)
print("Test label counts:", test_label_counts)


In [ ]:
for images, labels in train_ds.take(1):
    print("image batch:", images.shape)
    print("label batch:", labels.shape)
    print("labels:", labels[:8].numpy().reshape(-1))


In [ ]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.10),
        keras.layers.RandomZoom(0.20),
        keras.layers.RandomTranslation(0.05, 0.05),
        keras.layers.RandomContrast(0.20),
    ],
    name="data_augmentation",
)


In [ ]:
def build_vit_model():
    backbone = keras_hub.models.ViTBackbone.from_preset(
        "vit_base_patch16_224_imagenet21k"
    )

    preprocessing = keras.Sequential(
        [keras.layers.Rescaling(1.0 / 255)],
        name="preprocessing",
    )

    inputs = keras.Input(shape=img_size + (3,))
    x = preprocessing(inputs)
    x = data_augmentation(x)
    x = backbone(x)
    x = x[:, 0, :]
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="cancer_prob")(x)

    model = keras.Model(inputs, outputs)
    return model, backbone


def compile_binary_model(model, learning_rate, weight_decay=None):
    if weight_decay is None:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )


def find_transformer_blocks(backbone):
    all_backbone_layers = list(backbone._flatten_layers(include_self=False, recursive=True))
    transformer_blocks = [
        layer for layer in all_backbone_layers
        if "tranformer_block" in layer.name or "transformer_block" in layer.name
    ]
    if not transformer_blocks:
        candidate_names = [
            layer.name for layer in all_backbone_layers
            if any(token in layer.name.lower() for token in ["vit", "patch", "encoder", "block"])
        ]
        raise ValueError(
            "No transformer blocks found in backbone. Candidates: " + ", ".join(candidate_names[:30])
        )
    return all_backbone_layers, transformer_blocks


In [ ]:
stage1_model, stage1_backbone = build_vit_model()
stage1_backbone.trainable = False
compile_binary_model(stage1_model, learning_rate=1e-4)

print("Backbone type:", type(stage1_backbone).__name__)
print("Model output shape:", stage1_model.output_shape)
print("Stage 1 trainable weights:", len(stage1_model.trainable_weights))

stage1_ckpt = "/mnt/e/SW_training_outputs/checkpoints/best_vit_stage1_comparison.keras"
cb1 = [
    keras.callbacks.ModelCheckpoint(
        stage1_ckpt,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

history1 = stage1_model.fit(
    train_ds_prep,
    validation_data=val_ds_prep,
    epochs=10,
    callbacks=cb1,
    verbose=1,
)


In [ ]:
unfreeze_candidates = [2, 4, 6]
comparison_results = []
comparison_histories = {}
comparison_roc = {}

for num_blocks_to_unfreeze in unfreeze_candidates:
    print(f"\n===== Stage 2 experiment: unfreeze last {num_blocks_to_unfreeze} transformer blocks =====")

    stage2_model = keras.models.load_model(stage1_ckpt, compile=False)
    stage2_backbone = next(
        layer for layer in stage2_model.layers
        if isinstance(layer, keras_hub.models.ViTBackbone)
    )

    stage2_backbone.trainable = True
    all_backbone_layers, transformer_blocks = find_transformer_blocks(stage2_backbone)
    for layer in all_backbone_layers:
        layer.trainable = False

    unfrozen_blocks = transformer_blocks[-num_blocks_to_unfreeze:]
    for layer in unfrozen_blocks:
        layer.trainable = True

    patching_layer = stage2_backbone.get_layer("vit_patching_and_embedding")
    print("Found transformer blocks:", len(transformer_blocks))
    print("Unfrozen transformer blocks:", [layer.name for layer in unfrozen_blocks])
    print("Patching/embedding trainable:", patching_layer.trainable)
    print("Stage 2 trainable weights:", len(stage2_model.trainable_weights))

    compile_binary_model(stage2_model, learning_rate=1e-5, weight_decay=1e-4)

    ckpt_path = f"/mnt/e/SW_training_outputs/checkpoints/best_vit_unfreeze_{num_blocks_to_unfreeze}.keras"
    cb2 = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    history2 = stage2_model.fit(
        train_ds_prep,
        validation_data=val_ds_prep,
        epochs=25,
        callbacks=cb2,
        verbose=1,
    )
    comparison_histories[num_blocks_to_unfreeze] = history2

    best_model = keras.models.load_model(ckpt_path, compile=False)
    compile_binary_model(best_model, learning_rate=1e-5, weight_decay=1e-4)
    test_metrics = best_model.evaluate(test_ds_prep, return_dict=True, verbose=1)
    test_true, test_prob = collect_predictions(test_ds_prep, best_model)
    test_pred = (test_prob >= 0.5).astype("int32")
    test_auc = roc_auc_score(test_true, test_prob)
    test_acc = accuracy_score(test_true, test_pred)
    test_fpr, test_tpr, _ = roc_curve(test_true, test_prob)

    comparison_roc[num_blocks_to_unfreeze] = {
        "fpr": test_fpr,
        "tpr": test_tpr,
        "auc": test_auc,
    }
    comparison_results.append(
        {
            "unfreeze_blocks": num_blocks_to_unfreeze,
            "ckpt_path": ckpt_path,
            "best_val_auc": float(max(history2.history["val_auc"])),
            "best_val_acc": float(max(history2.history["val_acc"])),
            "best_val_loss": float(min(history2.history["val_loss"])),
            "test_auc": float(test_auc),
            "test_acc": float(test_acc),
            "test_loss": float(test_metrics["loss"]),
            "test_precision": float(test_metrics["precision"]),
            "test_recall": float(test_metrics["recall"]),
        }
    )

comparison_results = sorted(comparison_results, key=lambda item: item["unfreeze_blocks"])
best_result = max(comparison_results, key=lambda item: item["test_auc"])
best_unfreeze = best_result["unfreeze_blocks"]
best_ckpt_path = best_result["ckpt_path"]
print("\nBest setting by test AUC:", best_result)


In [ ]:
print("ViT unfreeze comparison summary:")
for result in comparison_results:
    print(
        f"unfreeze={result['unfreeze_blocks']:>2} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_auc={result['best_val_auc']:.4f} | "
        f"test_loss={result['test_loss']:.4f} | "
        f"test_acc={result['test_acc']:.4f} | "
        f"test_auc={result['test_auc']:.4f}"
    )


In [ ]:
block_values = [item["unfreeze_blocks"] for item in comparison_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for depth in block_values:
    history = comparison_histories[depth].history
    axes[0].plot(history["loss"], label=f"Train {depth}", alpha=0.85)
    axes[0].plot(history["val_loss"], linestyle="--", label=f"Val {depth}", alpha=0.85)
    axes[1].plot(history["acc"], label=f"Train {depth}", alpha=0.85)
    axes[1].plot(history["val_acc"], linestyle="--", label=f"Val {depth}", alpha=0.85)
    axes[2].plot(history["auc"], label=f"Train {depth}", alpha=0.85)
    axes[2].plot(history["val_auc"], linestyle="--", label=f"Val {depth}", alpha=0.85)

axes[0].set_title("Loss Curves by Unfreeze Depth")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Accuracy Curves by Unfreeze Depth")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True, alpha=0.3)

axes[2].set_title("AUC Curves by Unfreeze Depth")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUC")
axes[2].grid(True, alpha=0.3)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
metric_specs = [
    ("best_val_loss", "Best Val Loss", "tab:red"),
    ("best_val_acc", "Best Val Accuracy", "tab:green"),
    ("best_val_auc", "Best Val AUC", "tab:blue"),
]
labels = [str(v) for v in block_values]

for ax, (metric_name, title, color) in zip(axes, metric_specs):
    values = [item[metric_name] for item in comparison_results]
    bars = ax.bar(labels, values, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Unfrozen transformer blocks")
    ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

axes[0].set_ylabel("Score")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
test_metric_specs = [
    ("test_loss", "Test Loss", "tab:red"),
    ("test_acc", "Test Accuracy", "tab:green"),
    ("test_auc", "Test AUC", "tab:blue"),
]
labels = [str(v) for v in block_values]

for ax, (metric_name, title, color) in zip(axes, test_metric_specs):
    values = [item[metric_name] for item in comparison_results]
    bars = ax.bar(labels, values, color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Unfrozen transformer blocks")
    ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

axes[0].set_ylabel("Score")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
for depth in block_values:
    roc_info = comparison_roc[depth]
    plt.plot(
        roc_info["fpr"],
        roc_info["tpr"],
        label=f"Unfreeze {depth} (AUC = {roc_info['auc']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Comparison")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
best_model = keras.models.load_model(best_ckpt_path, compile=False)
compile_binary_model(best_model, learning_rate=1e-5, weight_decay=1e-4)
print("Loaded best comparison model from:", best_ckpt_path)
print("Best unfreeze depth:", best_unfreeze)

test_metrics = best_model.evaluate(test_ds_prep, return_dict=True)
print("Evaluation metrics:", test_metrics)

test_true, test_prob = collect_predictions(test_ds_prep, best_model)
test_pred = (test_prob >= 0.5).astype("int32")
test_fpr, test_tpr, _ = roc_curve(test_true, test_prob)
test_auc = roc_auc_score(test_true, test_prob)
test_acc = accuracy_score(test_true, test_pred)
cm = confusion_matrix(test_true, test_pred)

print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", classification_report(test_true, test_pred, target_names=class_names))

plt.figure(figsize=(6, 6))
plt.plot(test_fpr, test_tpr, label=f"ROC curve (AUC = {test_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (Best Depth = {best_unfreeze})")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()
